In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

base_url = "http://books.toscrape.com/catalogue/page-{}.html"

books = []

for page in range(1, 26):   # 25 pages × 20 books = 500 rows
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    items = soup.select("article.product_pod")

    for item in items:
        title = item.h3.a["title"]
        price = item.select_one("p.price_color").text.strip()
        availability = item.select_one("p.instock.availability").text.strip()
        rating = item.p["class"][1]

        link = item.h3.a["href"]
        book_url = "http://books.toscrape.com/catalogue/" + link

        product_page = requests.get(book_url)
        product_soup = BeautifulSoup(product_page.text, "html.parser")

        category = product_soup.select("ul.breadcrumb li a")[-1].text.strip()

        description = ""
        desc_tag = product_soup.select_one("#product_description ~ p")
        if desc_tag:
            description = desc_tag.text.strip()

        table = product_soup.select("table.table.table-striped tr")

        product_info = {}
        for row in table:
            key = row.th.text.strip()
            value = row.td.text.strip()
            product_info[key] = value

        books.append({
            "Title": title,
            "Category": category,
            "Price": price,
            "Availability": availability,
            "Rating": rating,
            "UPC": product_info.get("UPC"),
            "Price_excl_tax": product_info.get("Price (excl. tax)"),
            "Price_incl_tax": product_info.get("Price (incl. tax)"),
            "Number_of_reviews": product_info.get("Number of reviews"),
            "URL": book_url
        })

    print(f"Page {page} done, total books: {len(books)}")
    time.sleep(1)

df = pd.DataFrame(books)

print(df.shape)
print(df.head())

df.to_csv("books_500_rows.csv", index=False)

Page 1 done, total books: 20
Page 2 done, total books: 40
Page 3 done, total books: 60
Page 4 done, total books: 80
Page 5 done, total books: 100
Page 6 done, total books: 120
Page 7 done, total books: 140
Page 8 done, total books: 160
Page 9 done, total books: 180
Page 10 done, total books: 200
Page 11 done, total books: 220
Page 12 done, total books: 240
Page 13 done, total books: 260
Page 14 done, total books: 280
Page 15 done, total books: 300
Page 16 done, total books: 320
Page 17 done, total books: 340
Page 18 done, total books: 360
Page 19 done, total books: 380
Page 20 done, total books: 400
Page 21 done, total books: 420
Page 22 done, total books: 440
Page 23 done, total books: 460
Page 24 done, total books: 480
Page 25 done, total books: 500
(500, 10)
                                   Title            Category    Price  \
0                   A Light in the Attic              Poetry  Â£51.77   
1                     Tipping the Velvet  Historical Fiction  Â£53.74   
2        